# Non-linear static analysis with the Newton-Raphson method

A non-linear static analysis with [Analysis](../analysis.rst) requires
four callables, which calculate the external force vector, the internal
force vector, the constitutive stiffness matrix and the geometric stiffness
matrix. The tangent stiffness matrix
$\boldsymbol{K}_T = \boldsymbol{K}_C + \boldsymbol{K}_G$ should be the exact
derivative of the internal force vector, which gives quadratic convergence
to the Newton-Raphson method, described in
[newton_raphson._solver_NR](../newton_raphson.rst).

The class below, of `tests/analytic_problems.py`, used in the `structsolve`
unit tests, wraps analytic problems into these callables:

In [1]:
import os
import sys

# the helper functions of the tests are imported from the folder tests/
sys.path.insert(0, os.path.join('..', 'tests'))

import numpy as np
from scipy.sparse import csc_matrix

from structsolve import Analysis

from analytic_problems import convergence_orders
from test_newton_raphson import check_increments_and_equilibrium

In [2]:
class Problem(object):
    """Wrap analytic callbacks and record every call to ``calc_fint``

    Each recorded call stores the number of increments already accepted by
    the solver, allowing the iterations of each step to be counted.

    """
    def __init__(self, fext, fint, kC, kG):
        self.fext = fext
        self.fint = fint
        self.kC = kC
        self.kG = kG
        self.an = None
        self.fint_calls = []

    def calc_fext(self, inc=1., silent=True):
        return inc*self.fext

    def calc_fint(self, c, silent=True):
        self.fint_calls.append((len(self.an.increments), c.copy()))
        return self.fint(c)

    def calc_kC(self, c=None, NLgeom=False, silent=True):
        return csc_matrix(self.kC)

    def calc_kG(self, c=None, NLgeom=False, silent=True):
        return csc_matrix(self.kG(c))

    def analysis(self, **kwargs):
        an = Analysis(calc_fext=self.calc_fext, calc_fint=self.calc_fint,
                      calc_kC=self.calc_kC, calc_kG=self.calc_kG)
        for k, v in kwargs.items():
            setattr(an, k, v)
        self.an = an
        return an

    def calls_per_step(self):
        """Number of ``calc_fint`` calls spent to reach each accepted step"""
        steps = [s for s, _ in self.fint_calls]
        return [steps.count(i) for i in range(len(self.an.increments))]

    def solve_reference(self, lbd=1., c0=None, tol=1e-15):
        """Plain dense Newton-Raphson to obtain a reference solution"""
        c = np.linalg.solve(self.kC, lbd*self.fext) if c0 is None else c0
        for _ in range(100):
            R = self.fint(c) - lbd*self.fext
            if np.linalg.norm(R) <= tol*np.linalg.norm(lbd*self.fext):
                return c
            c = c + np.linalg.solve(self.kC + self.kG(c), -R)
        raise RuntimeError('reference solution did not converge')

One of these problems is a chain of springs with a cubic hardening, where
$\boldsymbol{F}_{int} = \boldsymbol{K} \boldsymbol{u} + a \boldsymbol{u}^3$:

In [3]:
def hardening_springs(P=50.):
    """Chain of 4 springs with cubic hardening: ``fint = K u + a u**3``"""
    n = 4
    K = 100.*(2*np.eye(n) - np.eye(n, k=1) - np.eye(n, k=-1))
    K[-1, -1] = 100.
    a = 50.*np.arange(1, n+1)
    fext = np.zeros(n)
    fext[-1] = P
    return Problem(fext, lambda u: K @ u + a*u**3, K,
                   lambda u: np.diag(3*a*u**2))

The full load is applied in a single step, starting from the linear
solution, and the order of convergence of the iterations is verified, in the
test `test_nr_full_load_quadratic_convergence` of
`tests/test_newton_raphson.py`:

In [4]:
def test_nr_full_load_quadratic_convergence():
    p = hardening_springs()
    p.analysis(initialInc=1.)
    p.an.static(NLgeom=True, silent=True)
    check_increments_and_equilibrium(p)
    assert len(p.an.increments) == 1

    ref = p.solve_reference()
    errors = [np.linalg.norm(c - ref)/np.linalg.norm(ref)
              for _, c in p.fint_calls]
    orders = convergence_orders(errors)
    assert len(orders) >= 1, errors
    assert min(orders) > 1.6, (orders, errors)
    # plain Newton from the linear predictor needs 9 residual evaluations to
    # reach machine precision; relTOL requires at most that
    assert len(p.fint_calls) <= 9

In [5]:
test_nr_full_load_quadratic_convergence()
print('test_nr_full_load_quadratic_convergence: passed')

test_nr_full_load_quadratic_convergence: passed


The analysis can also be performed in increments, with `initialInc`
defining the size of the first load increment:

In [6]:
def test_nr_incremental_hardening():
    p = hardening_springs()
    p.analysis(initialInc=0.1)
    p.an.static(NLgeom=True, silent=True)
    check_increments_and_equilibrium(p)
    # a good predictor with an exact tangent needs few iterations per step
    assert max(p.calls_per_step()) <= 5, p.calls_per_step()

In [7]:
test_nr_incremental_hardening()
print('test_nr_incremental_hardening: passed')

test_nr_incremental_hardening: passed
